In [1]:
import pandas as pd
from collections import defaultdict

In [3]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

%matplotlib inline  
from matplotlib import rcParams
rcParams['figure.figsize'] = (16, 100)

import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore") # Ignore all warnings
# warnings.filterwarnings("ignore", category=RRuntimeWarning) # Show some warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
%%javascript
// Disable auto-scrolling
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

In [5]:
%%R

require('tidyverse')
require('DescTools')

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Loading required package: tidyverse
Loading required package: DescTools


In [21]:
df = pd.read_csv("atp_matches_2023.csv")

In [37]:
%%R

get_serve_win_prob_safe <- function(first_in, first_won, second_won, total_pts) {
  if (is.na(first_in) || is.na(first_won) || is.na(second_won) || is.na(total_pts) ||
      first_in == 0 || total_pts == 0 || (total_pts - first_in) == 0) {
    return(NA_real_)
  }

  first_pct <- first_in / total_pts
  first_win_pct <- first_won / first_in
  second_pts <- total_pts - first_in
  second_win_pct <- second_won / second_pts

  p_win <- (first_pct * first_win_pct) + ((1 - first_pct) * second_win_pct)
  return(p_win)
}


In [38]:
%%R -i df

df$w_1stIn <- as.numeric(df$w_1stIn)
df$w_1stWon <- as.numeric(df$w_1stWon)
df$w_2ndWon <- as.numeric(df$w_2ndWon)
df$w_svpt <- as.numeric(df$w_svpt)

df$l_1stIn <- as.numeric(df$l_1stIn)
df$l_1stWon <- as.numeric(df$l_1stWon)
df$l_2ndWon <- as.numeric(df$l_2ndWon)
df$l_svpt <- as.numeric(df$l_svpt)


In [39]:
%%R

library(purrr)

df$p_win_winner <- pmap_dbl(
  list(df$w_1stIn, df$w_1stWon, df$w_2ndWon, df$w_svpt),
  get_serve_win_prob_safe
)

df$p_win_loser <- pmap_dbl(
  list(df$l_1stIn, df$l_1stWon, df$l_2ndWon, df$l_svpt),
  get_serve_win_prob_safe
)


In [40]:
%%R
summary(df$p_win_winner)
summary(df$p_win_loser)


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
 0.0000  0.5429  0.5902  0.5867  0.6364  0.9231     171 
